In [2]:
!pip install matplotlib scipy pandas cvxpy tqdm seaborn openai cvxpy[glpk] polarix -q

zsh:1: no matches found: cvxpy[glpk]


In [3]:
import pickle
import sys
import time                                                                                                                                    
from pathlib import Path
from collections import Counter, defaultdict                                                                                                   
from itertools import combinations

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.optimize import linprog
import pandas as pd

sys.path.insert(0, "/Users/gabesmithline/Desktop/Causal-Game-Analysis")


from src.iterative_game_analysis.metagame import MetaGame

from visuals.visualize_analysis import DISPLAY_NAMES, STRATEGY_ORDER
from evaluation.curb_analysis import *
from evaluation.bootstrap_analysis import load_all_games, compute_payoff_matrix_from_games 
from src.iterative_game_analysis.full_analysis import load_crossplay_to_dataframe                                                              
from src.iterative_game_analysis.bootstrap import Bootstrap 


/Users/gabesmithline/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/matplotlib-tojou2s9 because there was an issue with the default path (/Users/gabesmithline/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


In [4]:

crossplay_dir = "/Users/gabesmithline/Desktop/Causal-Game-Analysis/data/crossplay"
print(crossplay_dir)
strategy_names = ["walk", "tough", "soft",
                    "openai_5.2_none", "openai_5.2_low", "openai_5.4_low", "ef1_bargainer", "ppo", "psro", "nfsp",
                    "openai_5.4_medium", "openai_5.2_medium"] 
df = load_crossplay_to_dataframe(crossplay_dir, strategy_names, raw_utility=True)  
print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")

/Users/gabesmithline/Desktop/Causal-Game-Analysis/data/crossplay
  Loading walk vs walk...
  Loading walk vs tough...
  Loading walk vs soft...
  Loading walk vs openai_5.2_none...
  Loading walk vs openai_5.2_low...
  Loading walk vs openai_5.4_low...
  Loading walk vs ef1_bargainer...
  Loading walk vs ppo...
  Loading walk vs psro...
  Loading walk vs nfsp...
  Loading walk vs openai_5.4_medium...
  Loading walk vs openai_5.2_medium...
  Loading tough vs walk...
  Loading tough vs tough...
  Loading tough vs soft...
  Loading tough vs openai_5.2_none...
  Loading tough vs openai_5.2_low...
  Loading tough vs openai_5.4_low...
  Loading tough vs ef1_bargainer...
  Loading tough vs ppo...
  Loading tough vs psro...
  Loading tough vs nfsp...
  Loading tough vs openai_5.4_medium...
  Loading tough vs openai_5.2_medium...
  Loading soft vs walk...
  Loading soft vs tough...
  Loading soft vs soft...
  Loading soft vs openai_5.2_none...
  Loading soft vs openai_5.2_low...
  Loading soft 

In [5]:
# Build the average matrices (no bootstrap, just the point estimate)
boot = Bootstrap(df=df, n_samples=1, seed=42, policies=strategy_names)
matrices = boot._build_all_matrices(df, strategy_names)

avg_payoff = matrices["payoff"]       # 10x10, used for equilibrium solving + UW
nw_matrix = matrices["nw"]            # 10x10, Nash welfare per matchup
nw_plus_matrix = matrices["nw_plus"]  # 10x10, Nash welfare on advantages
ef1_matrix = np.nan_to_num(matrices["ef1"])          # 10x10, EF1 frequency per matchup
ef1_plus_matrix = np.nan_to_num(matrices["ef1_plus"])# 10x10, EF1+ frequency per matchup  


print("Empirical Meta-Game (avg payoff):\n")
header = "".join(f"{s[:8]:>10}" for s in strategy_names)
print(f"{'':>10}{header}")
print("-" * (10 + 10 * len(strategy_names)))
for i, name in enumerate(strategy_names):
    row = "".join(f"{avg_payoff[i, j]:>10.2f}" for j in range(len(strategy_names)))
    print(f"{name[:8]}{row}")

Empirical Meta-Game (avg payoff):

                walk     tough      soft  openai_5  openai_5  openai_5  ef1_barg       ppo      psro      nfsp  openai_5  openai_5
----------------------------------------------------------------------------------------------------------------------------------
walk    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81
tough    303.81    303.81    579.17    330.69    324.55    326.84    304.62    319.51    317.67    325.16    311.83    320.89
soft    303.81     50.43    303.59    230.81    229.89    231.06    310.96    197.45    165.56    287.51    240.42    246.77
openai_5    303.81    298.73    446.43    381.55    382.81    382.91    372.72    358.69    343.24    357.85    390.83    386.70
openai_5    303.81    302.70    438.95    383.37    382.31    398.28    377.30    364.88    364.36    368.21    393.20    380.39
openai_5    303.81    299.88    444.60    393.23    378.27    379.37 

In [16]:

sigma = MetaGame(payoff_matrix=avg_payoff.T, policies=strategy_names).solve("mene")
for mass, name in zip(sigma, strategy_names):
    print(f"Strat name: {name}, mass: {mass}")

Strat name: walk, mass: -0.0
Strat name: tough, mass: -0.0
Strat name: soft, mass: 0.8130579090460317
Strat name: openai_5.2_none, mass: 0.0
Strat name: openai_5.2_low, mass: -0.0
Strat name: openai_5.4_low, mass: 0.0
Strat name: ef1_bargainer, mass: 0.1869420909539683
Strat name: ppo, mass: -0.0
Strat name: psro, mass: -0.0
Strat name: nfsp, mass: -0.0
Strat name: openai_5.4_medium, mass: -0.0
Strat name: openai_5.2_medium, mass: -0.0


In [7]:
nw_plus_mene = sigma @ nw_plus_matrix @ sigma
print(nw_plus_mene)
nw_mene= sigma @ nw_matrix @ sigma 
print(nw_mene)
ef1__mene = sigma @ ef1_matrix @ sigma
print(ef1__mene)
ef1_plus_mene = sigma @ ef1_plus_matrix @ sigma 
print(ef1_plus_mene)

51.24416235697045
325.9094850257247
0.5123296521922903
0.5293164111949451


In [8]:
METRIC_NAMES = ["uw", "nw", "nw_plus", "ef1", "ef1_plus"]                                                                                                                 
metric_matrices = {
    "uw": avg_payoff, "nw": nw_matrix, "nw_plus": nw_plus_matrix,                                                                                                         
    "ef1": ef1_matrix, "ef1_plus": ef1_plus_matrix,
}
policy_to_idx = {p: i for i, p in enumerate(strategy_names)}
ablatable = [s for s in strategy_names if s != "walk"]

# Full-game welfare
W_full = {m: float(sigma @ np.nan_to_num(metric_matrices[m], nan=0.0) @ sigma) for m in METRIC_NAMES}
print("Full-game welfare:", {m: f"{v:.4f}" for m, v in W_full.items()})

# Hold-one-out
singleton_effects = {}
for s in ablatable:
    remaining = [q for q in strategy_names if q != s]
    idx = [policy_to_idx[q] for q in remaining]
    sub_mg = MetaGame(remaining, avg_payoff[np.ix_(idx, idx)])
    sigma_sub = sub_mg.solve("mene")
    W_sub = {m: float(sigma_sub @ np.nan_to_num(metric_matrices[m][np.ix_(idx, idx)], nan=0.0) @ sigma_sub) for m in METRIC_NAMES}
    singleton_effects[s] = {m: (W_full[m] - W_sub[m]) / W_full[m] * 100  for m in METRIC_NAMES}   

df_single = pd.DataFrame(singleton_effects).T
df_single = df_single.sort_values("uw", key=abs, ascending=False)
df_single


Full-game welfare: {'uw': '367.3338', 'nw': '325.9095', 'nw_plus': '51.2442', 'ef1': '0.5123', 'ef1_plus': '0.5293'}


/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:285: UserWarning: Replicator dynamics found solution with entropy=0.0000, regret=0.000000 (from 40/41 valid starts).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:87: UserWarning: MILP failed (Solution has regret 0.002551 > 1e-05). Replicator dynamics succeeded at tolerance 1e-05.
  warnings.warn(


,uw,nw,nw_plus,ef1,ef1_plus
psro,-5.320019e+00,-7.499577e+00,-2.722893e+01,-3.257864e+01,-5.250853e+01
openai_5.2_low,-1.479062e+00,-2.331449e+00,-8.608584e+00,-8.253193e+00,-1.433644e+01
ppo,3.663779e-01,1.005478e+00,-8.482145e-01,-4.156039e+00,-4.354476e+00
openai_5.2_medium,4.951871e-13,5.058028e-13,5.269015e-13,4.984121e-13,5.033918e-13
openai_5.4_medium,4.487633e-13,4.709198e-13,4.991698e-13,4.550719e-13,4.824171e-13
tough,4.332887e-13,4.360369e-13,4.437065e-13,4.550719e-13,4.614425e-13
openai_5.4_low,4.178141e-13,4.360369e-13,4.159748e-13,4.117317e-13,4.194931e-13
openai_5.2_none,3.713903e-13,3.662710e-13,4.437065e-13,4.117317e-13,4.194931e-13
nfsp,3.713903e-13,3.662710e-13,4.437065e-13,4.117317e-13,4.194931e-13
soft,3.249665e-13,3.313880e-13,3.189140e-13,3.250514e-13,3.355945e-13


In [9]:
welfare_cache = {}

def get_welfare(exclude_set):
    key = frozenset(exclude_set)
    if key not in welfare_cache:
        remaining = [q for q in strategy_names if q not in exclude_set]
        idx = [policy_to_idx[q] for q in remaining]
        sub_mg = MetaGame(remaining, avg_payoff[np.ix_(idx, idx)])
        sig = sub_mg.solve("mene")
        welfare_cache[key] = {
            m: float(sig @ np.nan_to_num(metric_matrices[m][np.ix_(idx, idx)], nan=0.0) @ sig)
            for m in METRIC_NAMES
        }
    return welfare_cache[key]

welfare_cache[frozenset()] = W_full
for s in ablatable:
    get_welfare({s})

pairs = list(combinations(ablatable, 2))
pair_effects = {}
for a, b in pairs:
    W_no_a = get_welfare({a})
    W_no_b = get_welfare({b})
    W_no_ab = get_welfare({a, b})
    pair_effects[(a, b)] = {m: W_full[m] - W_no_a[m] - W_no_b[m] + W_no_ab[m] for m in METRIC_NAMES}

df_pairs = pd.DataFrame(pair_effects).T
df_pairs.index = [f"{a} x {b}" for a, b in pair_effects.keys()]
df_pairs = df_pairs.sort_values("uw", key=abs, ascending=False)
df_pairs

/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:285: UserWarning: Replicator dynamics found solution with entropy=0.0000, regret=0.000000 (from 40/41 valid starts).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:87: UserWarning: MILP failed (Solution has regret 0.002551 > 1e-05). Replicator dynamics succeeded at tolerance 1e-05.
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000011 > 1e-05).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.001 (original failure: Solution has regret 0.000567 > 1e-05).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_g

,uw,nw,nw_plus,ef1,ef1_plus
psro x openai_5.4_medium,-8.306277e+01,-9.136772e+01,-6.519740e+01,-6.792397e-01,-8.072527e-01
soft x ppo,-6.217472e+01,-6.364894e+01,-5.167882e+01,-5.336223e-01,-5.523654e-01
openai_5.2_none x ppo,-6.217472e+01,-6.364894e+01,-5.167882e+01,-5.336223e-01,-5.523654e-01
openai_5.4_low x ppo,-6.217472e+01,-6.364894e+01,-5.167882e+01,-5.336223e-01,-5.523654e-01
ppo x openai_5.4_medium,-6.217472e+01,-6.364894e+01,-5.167882e+01,-5.336223e-01,-5.523654e-01
ppo x openai_5.2_medium,-6.217472e+01,-6.364894e+01,-5.167882e+01,-5.336223e-01,-5.523654e-01
tough x ppo,-6.217472e+01,-6.364894e+01,-5.167882e+01,-5.336223e-01,-5.523654e-01
ppo x nfsp,-6.217472e+01,-6.364894e+01,-5.167882e+01,-5.336223e-01,-5.523654e-01
ef1_bargainer x ppo,-6.217472e+01,-6.364894e+01,-5.167882e+01,-5.336223e-01,-5.523654e-01
openai_5.2_low x ppo,1.624573e+01,2.031221e+01,9.022791e+00,9.182550e-02,1.390126e-01
